# SAM confound diagnostic — follow-up: mask QC, robust area statistics, within-species shape, Algorithm 2

Re-runs the first SAM run (`sam_confound.ipynb`) with the **same sampling and mask choice**, but keeps every per-image
value so that the following can be computed (all fixed before this run):

1. **Primary (as pre-specified):** mean-area ratio, shape-only AUC, attention collapse, on all masks — must reproduce the first run.
2. **Class-blind mask QC:** SAM's own default quality filter (predicted IoU ≥ 0.88, stability ≥ 0.95); secondary: also drop masks
   touching the frame edge. Exclusion rate reported **per class** so a biased filter is visible.
3. **Median-area ratio** (exploratory) with bootstrap CIs next to the mean.
4. **TeaLeafDiseaseBD only:** the subset where SAM and Otsu agree (IoU ≥ 0.80) — Otsu-conditioned, diagnostic only.
5. **Species:** PlantVillage and PlantDoc shape-only AUC within each species, after removing species means, and from species alone.
6. **Algorithm 2 verdict** per dataset, with C3 (A_match on shape-matched pairs) now computed.

Output: `sam_confound_followup.json` + `sam_confound_per_image.npz` — upload both. ~2 h on a GPU (same as the first run);
cells 5–6 are CPU and can be re-run from the .npz.

In [ ]:
# CELL 0 — CONFIG
DRIVE      = '/content/drive/MyDrive'
TEA_ROOT   = f'{DRIVE}/Tea Leaf Dataset'
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
CVD        = f'{STUDY}/cross_validation_datasets'
OUT        = f'{RES}/sam_confound_followup.json'
NPZ        = f'{RES}/sam_confound_per_image.npz'
FIRST_RUN  = f'{RES}/sam_confound.json'          # the first run: used only as a reproduction check
LOCAL_DATA = '/tmp/tea_leaf_local'
N_PER_CLASS, SEED = 200, 0              # as in the original run (notebook 08)
SAM_MAX_SIDE, SAM_AREA = 1024, (0.03, 0.90)
MIN_MASK_RATE = 0.70
PLANTDOC_HEALTHY = ["apple leaf", "bell_pepper leaf", "blueberry leaf", "cherry leaf",
                    "raspberry leaf", "soyabean leaf", "tomato leaf", "grape leaf"]
DATASETS = {"TeaLeafDiseaseBD": (LOCAL_DATA, ("healthy",)),
            "PlantVillage": (f"{CVD}/PlantVillage", ("healthy",)),
            "RiceLeafBD": (f"{CVD}/RiceLeafDiseaseBD/Original images", ("healthy",)),
            "RoCoLe": (f"{CVD}/RoCoLe_binary", ("healthy",)),
            "PlantDoc (field)": (f"{CVD}/PlantDoc", tuple(PLANTDOC_HEALTHY))}
# the published Otsu results (artifacts/cross_dataset_confound.json), for the old-vs-new table
OTSU = {"TeaLeafDiseaseBD": (1.936, 0.834, 0.928, 0.543), "PlantVillage": (1.018, 0.574, 0.611, 0.574),
        "RiceLeafBD": (1.060, 0.623, 0.730, 0.719), "RoCoLe": (0.922, 0.572, 0.727, 0.750), "PlantDoc (field)": (1.074, 0.600, 0.644, 0.657)}
# ---- fixed BEFORE this run, class-blind, never tuned on results ----
QC_PRIMARY = dict(pred_iou=0.88, stability=0.95)   # SAM's own automatic-mask-generator defaults (Kirillov et al., 2023)
QC_BORDER  = True                                  # secondary QC: additionally drop masks touching the frame edge
N_BOOT     = 2000
MIN_PER_SPECIES = 20                                # per class, for a within-species AUC


In [ ]:
# CELL 1 — setup, SAM checkpoint, original modules, local copy of TeaLeafDiseaseBD
from google.colab import drive; drive.mount('/content/drive')
!pip -q install timm git+https://github.com/facebookresearch/segment-anything.git
!wget -q -nc --no-check-certificate https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -P /content/
import os, sys, io, json, glob, time, random, shutil, hashlib, warnings
import numpy as np, cv2, torch, timm, torchvision.transforms as T
from PIL import Image, ImageDraw
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
for m in ('shape_confound.py',):
    hit = glob.glob(f'{STUDY}/**/{m}', recursive=True); assert hit, m + ' not found on Drive'
    shutil.copy(hit[0], f'/content/{m}')
sys.path.insert(0, '/content'); import shape_confound as sc
assert hasattr(sc, 'leaf_mask') and hasattr(sc, 'shape_features') and hasattr(sc, 'attn_scores')
def _cp(t):
    if not os.path.exists(t[1]): os.makedirs(os.path.dirname(t[1]), exist_ok=True); shutil.copyfile(*t)
tasks = [(os.path.join(r, f), os.path.join(LOCAL_DATA, os.path.relpath(r, TEA_ROOT), f)) for r, _, fs in os.walk(TEA_ROOT) for f in fs]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('TeaLeafDiseaseBD local copy:', len(tasks), 'files')


In [ ]:
# CELL 2 — DINO ViT-S/16 with the CLS-attention hook (verbatim from notebook 08)
dv1_path = os.path.join(STUDY, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    os.system('wget -q -O "%s" https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth' % dv1_path)
dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
dino_v1.load_state_dict({k.replace('module.', '').replace('backbone.', '').replace('encoder.', ''): v for k, v in ckpt.items() if 'head' not in k}, strict=False)
dino_v1.eval().to(device)
_dv1_store = {}; att = dino_v1.blocks[-1].attn
def _fwd(x, **kw):
    B, N, Cc = x.shape
    q, k, v = att.qkv(x).reshape(B, N, 3, att.num_heads, Cc // att.num_heads).permute(2, 0, 3, 1, 4).unbind(0)
    s = (Cc // att.num_heads) ** -0.5
    _dv1_store['cls_attn'] = ((q[:, :, 0:1, :] @ k.transpose(-2, -1)) * s).softmax(-1)[:, :, 0, 1:].detach().cpu()
    x = (((q @ k.transpose(-2, -1)) * s).softmax(-1) @ v).transpose(1, 2).reshape(B, N, Cc)
    return att.proj_drop(att.proj(x))
att.forward = _fwd
tf_dv1 = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
DV1_GRID = 14; topk = max(1, int(0.1 * DV1_GRID * DV1_GRID))
with torch.no_grad(): _ = dino_v1(torch.randn(1, 3, 224, 224).to(device))
print('attention hook OK', _dv1_store['cls_attn'].shape)


In [ ]:
# CELL 3 — SAM, centre-point prompt; identical mask choice to the first run, plus SAM's quality scores
from segment_anything import sam_model_registry, SamPredictor
from segment_anything.utils.amg import calculate_stability_score
_h = hashlib.md5(open('/content/sam_vit_b_01ec64.pth', 'rb').read()).hexdigest()
assert _h.startswith('01ec64'), 'SAM checkpoint checksum mismatch'
sam_model = sam_model_registry['vit_b'](checkpoint='/content/sam_vit_b_01ec64.pth').to(device).eval()
predictor = SamPredictor(sam_model)
def sam_mask(rgb):
    """Returns (mask, predicted_iou, stability, touches_border) or (None, ...). Mask choice exactly as the first run."""
    H, W = rgb.shape[:2]; s = min(1.0, SAM_MAX_SIDE / max(H, W))
    small = cv2.resize(rgb, (int(W * s), int(H * s)), interpolation=cv2.INTER_AREA) if s < 1 else rgb
    predictor.set_image(small); h, w = small.shape[:2]
    logits, scores, _ = predictor.predict(point_coords=np.array([[w / 2, h / 2]]), point_labels=np.array([1]),
                                          multimask_output=True, return_logits=True)
    stab = calculate_stability_score(torch.from_numpy(logits), sam_model.mask_threshold, 1.0).numpy()
    best, bs, bi = None, -1, -1
    for i, (lg, q) in enumerate(zip(logits, scores)):
        m = lg > sam_model.mask_threshold
        if SAM_AREA[0] <= m.mean() <= SAM_AREA[1] and q > bs: best, bs, bi = m, q, i
    if best is None: return None, np.nan, np.nan, False
    border = bool(best[0].any() or best[-1].any() or best[:, 0].any() or best[:, -1].any())
    return best, float(bs), float(stab[bi]), border


In [ ]:
# CELL 4 — scan every dataset ONCE and keep everything per image (same sampling as the first run)
def species_of(name, cls):
    if name == 'PlantVillage': return cls.split('_')[0].lower()
    if name.startswith('PlantDoc'): return cls.split(' ')[0].lower()
    return 'single'
def sample(root, hk):
    rng = random.Random(SEED); recs = []
    classes = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    for c in classes:
        fs = [f for f in os.listdir(os.path.join(root, c)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for f in rng.sample(fs, min(N_PER_CLASS, len(fs))):
            recs.append(dict(path=os.path.join(root, c, f), cls=c, y=0 if any(k in c.lower() for k in hk) else 1))
    return recs
PER = {}
def scan_raw(name, root, hk):
    recs = sample(root, hk); out = {k: [] for k in ('y', 'cls', 'species', 'area', 'shape', 'G', 'W', 'iou_pred', 'stab', 'border', 'iou_otsu')}
    t0 = time.time(); n_sampled = len(recs); n_nomask = 0
    for r in recs:
        try: img = Image.open(r['path']).convert('RGB')
        except Exception: continue
        m, q, st, bd = sam_mask(np.array(img))
        if m is None: n_nomask += 1; continue
        f = sc.shape_features(m)
        if f is None: n_nomask += 1; continue
        g, w = sc.attn_scores(img, dino_v1, tf_dv1, _dv1_store, device, DV1_GRID, [0, 1], topk, m)
        io = np.nan
        if name == 'TeaLeafDiseaseBD':
            mo = sc.leaf_mask(img)
            if mo is not None:
                ms = cv2.resize(m.astype(np.uint8), mo.shape[::-1], interpolation=cv2.INTER_NEAREST).astype(bool)
                io = float((ms & mo).sum() / max(1, (ms | mo).sum()))
        for k, v in zip(out, (r['y'], r['cls'], species_of(name, r['cls']), float(m.mean()), np.asarray(f, float), g, w, q, st, bd, io)):
            out[k].append(v)
    out = {k: np.array(v) for k, v in out.items()}; out['n_sampled'] = n_sampled; out['n_nomask'] = n_nomask
    print('%-18s sampled %d, masked %d, %.1f min' % (name, n_sampled, len(out['y']), (time.time() - t0) / 60))
    return out
for name, (root, hk) in DATASETS.items():
    if not os.path.isdir(root): print(name, 'NOT FOUND'); continue
    PER[name] = scan_raw(name, root, hk)
np.savez_compressed(NPZ, **{f'{d}|{k}': v for d, D in PER.items() for k, v in D.items()})
print('per-image data saved ->', NPZ, '(all later cells are CPU only and can be re-run from this file)')


In [ ]:
# CELL 5 — the analyses (CPU). Primary statistics exactly as pre-specified; everything else labelled.
dir_ = lambda a: max(a, 1 - a)
CV = StratifiedKFold(5, shuffle=True, random_state=0)
def shape_auc(S, Y):                                     # identical to the first run: max of LR and RF, 5-fold
    if len(set(Y)) < 2 or min((Y == 0).sum(), (Y == 1).sum()) < 5: return np.nan
    return float(max(cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), S, Y, cv=CV, scoring='roc_auc').mean(),
                     cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1), S, Y, cv=CV, scoring='roc_auc').mean()))
def match_pairs(SH, Y, tol_area=0.03, tol_aspect=0.20):   # as scripts/original/feature_shape_control.py
    area, aspect = SH[:, 0], SH[:, 1]; used, hh, dd = set(), [], []
    for i in np.where(Y == 0)[0]:
        cand = [j for j in np.where(Y == 1)[0] if j not in used and abs(area[j] - area[i]) < tol_area and abs(aspect[j] - aspect[i]) < tol_aspect]
        if cand:
            j = min(cand, key=lambda j: abs(area[j] - area[i]) + abs(aspect[j] - aspect[i])); used.add(j); hh.append(i); dd.append(j)
    return np.array(hh, int), np.array(dd, int)
def boot_ratio(A, Y, f, rng):
    h, d = A[Y == 0], A[Y == 1]; r = []
    for _ in range(N_BOOT): r.append(f(rng.choice(h, len(h))) / f(rng.choice(d, len(d))))
    return [float(np.percentile(r, 2.5)), float(np.percentile(r, 97.5))]
def axes(D, keep, label):
    Y, A, S, G, W = D['y'][keep], D['area'][keep], D['shape'][keep], D['G'][keep], D['W'][keep]
    ok = ~np.isnan(W); rng = np.random.default_rng(0)
    r = dict(subset=label, n=int(keep.sum()), n_healthy=int((Y == 0).sum()), n_diseased=int((Y == 1).sum()))
    if len(set(Y)) < 2: return r
    r.update(area_ratio_mean=float(A[Y == 0].mean() / A[Y == 1].mean()), area_ratio_mean_ci=boot_ratio(A, Y, np.mean, rng),
             area_ratio_median=float(np.median(A[Y == 0]) / np.median(A[Y == 1])), area_ratio_median_ci=boot_ratio(A, Y, np.median, rng),
             area_auc=float(dir_(roc_auc_score(Y, A))), shape_only_auc=shape_auc(S, Y),
             naive_attention_auc=float(dir_(roc_auc_score(Y, G))), within_leaf_attention_auc=float(dir_(roc_auc_score(Y[ok], W[ok]))))
    r['attention_collapse'] = r['naive_attention_auc'] - r['within_leaf_attention_auc']
    hh, dd = match_pairs(S, Y); r['n_matched_pairs'] = int(len(hh))
    if len(hh) >= 10:
        p = np.r_[hh, dd]; r['A_match'] = float(dir_(roc_auc_score(Y[p], G[p])))
    # Algorithm 2 as printed in the paper; "clearly exceeds" operationalised as >= 0.05 (stated in the JSON)
    if r['shape_only_auc'] >= 0.70 and r['attention_collapse'] >= 0.15: r['algorithm2'] = 'Confound present'
    elif r.get('A_match', -1) - r['shape_only_auc'] >= 0.05: r['algorithm2'] = 'Signal is pathology'
    else: r['algorithm2'] = 'Inconclusive; report both'
    return r
RES_ = {}
for name, D in PER.items():
    n = len(D['y']); allk = np.ones(n, bool)
    qc1 = (D['iou_pred'] >= QC_PRIMARY['pred_iou']) & (D['stab'] >= QC_PRIMARY['stability'])
    qc2 = qc1 & ~D['border']
    R = dict(mask_rate=float(n / D['n_sampled']), n_sampled=int(D['n_sampled']),
             excluded_by_qc={lab: {('healthy' if c == 0 else 'diseased'): float(1 - k[D['y'] == c].mean()) for c in (0, 1)}
                             for lab, k in (('qc_sam_defaults', qc1), ('qc_plus_border', qc2))},
             border_touch_rate={('healthy' if c == 0 else 'diseased'): float(D['border'][D['y'] == c].mean()) for c in (0, 1)},
             primary=axes(D, allk, 'all masks (pre-specified)'),
             qc_sam_defaults=axes(D, qc1, 'SAM-default quality filter (class-blind)'),
             qc_plus_border=axes(D, qc2, 'SAM-default filter + no border contact (class-blind)'))
    if name == 'TeaLeafDiseaseBD':
        io = D['iou_otsu']; agree = ~np.isnan(io) & (io >= 0.80)
        R['iou_with_otsu'] = dict(median=float(np.nanmedian(io)), p10=float(np.nanpercentile(io, 10)), frac_below_0_5=float(np.nanmean(io < 0.5)))
        R['otsu_agreement_subset'] = axes(D, agree, 'SAM/Otsu IoU >= 0.80 (Otsu-conditioned; diagnostic only)')
        bad = ~np.isnan(io) & (io < 0.5)
        R['gross_disagreement'] = dict(n=int(bad.sum()), mean_area_when_bad=float(D['area'][bad].mean()) if bad.any() else None,
                                       by_class={('healthy' if c == 0 else 'diseased'): float(bad[D['y'] == c].mean()) for c in (0, 1)})
    if name == 'PlantVillage' or name.startswith('PlantDoc'):          # is the shape signal a species artefact?
        sp = D['species']; per = {}
        both = np.zeros(n, bool)
        for s_ in np.unique(sp):
            k = sp == s_; nh, nd = int((D['y'][k] == 0).sum()), int((D['y'][k] == 1).sum())
            if min(nh, nd) >= MIN_PER_SPECIES:
                per[s_] = dict(n_healthy=nh, n_diseased=nd, shape_only_auc=shape_auc(D['shape'][k], D['y'][k])); both |= k
            else: per[s_] = dict(n_healthy=nh, n_diseased=nd, shape_only_auc=None)
        Sc = D['shape'].copy()
        for s_ in np.unique(sp[both]): Sc[both & (sp == s_)] -= Sc[both & (sp == s_)].mean(0)    # remove species means
        oh = (sp[both][:, None] == np.unique(sp[both])[None, :]).astype(float)
        w = [(v['n_healthy'] + v['n_diseased'], v['shape_only_auc']) for v in per.values() if v['shape_only_auc'] is not None]
        R['species'] = dict(per_species=per, n_species_with_both=int(len(w)),
                            shape_only_auc_same_images=shape_auc(D['shape'][both], D['y'][both]),
                            species_only_auc=shape_auc(oh, D['y'][both]),
                            shape_only_auc_species_centred=shape_auc(Sc[both], D['y'][both]),
                            weighted_within_species_auc=float(sum(a * b for a, b in w) / sum(a for a, _ in w)) if w else None)
    RES_[name] = R
# reproduction check against the first run (same sampling, same masks)
first = {r['dataset']: r for r in json.load(open(FIRST_RUN))['rows']}
for name, R in RES_.items():
    a, b = R['primary']['shape_only_auc'], first[name]['shape_only_auc']
    print('%-18s shape-only %.3f (first run %.3f)  collapse %+.3f (first run %+.3f)' % (name, a, b, R['primary']['attention_collapse'], first[name]['attention_collapse']))
    R['reproduces_first_run'] = bool(abs(a - b) < 0.02)
    if not R['reproduces_first_run']: print('   WARNING: %s does not reproduce the first run (|diff| >= 0.02) - report this' % name)


In [ ]:
# CELL 6 — readable summary + save
P = lambda x: ('%.3f' % x) if isinstance(x, float) and not np.isnan(x) else str(x)
for name, R in RES_.items():
    print('\n=== %s  (mask rate %.0f%%)' % (name, 100 * R['mask_rate']))
    print('   QC exclusion, healthy/diseased: SAM defaults %s | +border %s' % (R['excluded_by_qc']['qc_sam_defaults'], R['excluded_by_qc']['qc_plus_border']))
    for key in ('primary', 'qc_sam_defaults', 'qc_plus_border', 'otsu_agreement_subset'):
        if key not in R or 'shape_only_auc' not in R[key]: continue
        r = R[key]
        print('   %-52s n=%4d | ratio mean %s [%s-%s] median %s | shape %s | collapse %+.3f | A_match %s (%d pairs) | %s'
              % (r['subset'], r['n'], P(r['area_ratio_mean']), P(r['area_ratio_mean_ci'][0]), P(r['area_ratio_mean_ci'][1]), P(r['area_ratio_median']),
                 P(r['shape_only_auc']), r['attention_collapse'], P(r.get('A_match', np.nan)), r['n_matched_pairs'], r['algorithm2']))
    if 'species' in R:
        s = R['species']
        print('   species: shape-only on same images %s | species-only %s | species-centred shape %s | weighted within-species %s'
              % tuple(P(s[k]) for k in ('shape_only_auc_same_images', 'species_only_auc', 'shape_only_auc_species_centred', 'weighted_within_species_auc')))
        for sp_, v in s['per_species'].items():
            if v['shape_only_auc'] is not None: print('      %-12s %3d/%3d  shape-only %s' % (sp_, v['n_healthy'], v['n_diseased'], P(v['shape_only_auc'])))
json.dump(dict(results=RES_, qc_rules=dict(primary=QC_PRIMARY, secondary='primary + no frame-edge contact', fixed='before the run, class-blind'),
               algorithm2_note='printed rule: A_shape>=0.70 and collapse>=0.15 -> Confound present; "clearly exceeds" operationalised as A_match - A_shape >= 0.05',
               shape_auc_note='max of logistic regression and random forest, 5-fold CV, as in the first run',
               sam=dict(model='vit_b', prompt='centre point', max_side=SAM_MAX_SIDE, area=SAM_AREA)),
          open(OUT, 'w'), indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print('\nsaved', OUT, 'and', NPZ, '-> upload both')
